# Lab 4: Parallel Reduction

---

## Overview

This lab covers **parallel reduction**, one of the most fundamental and widely-used GPU programming patterns. Reduction operations—such as sum, max, min, and product—appear in countless algorithms from machine learning to scientific computing.

Unlike embarrassingly parallel problems, reduction has inherent **data dependencies**: computing a sum requires combining partial results. This lab teaches you how to efficiently parallelize such operations using **tree-based reduction** and **shared memory**.

Mastering reduction is essential because it forms the building block for more complex algorithms like histogram, softmax, batch normalization, and prefix sum.

## Learning Objectives

By the end of this lab, you will be able to:

1. Explain why naive atomic reduction is inefficient for large arrays
2. Implement tree-based parallel reduction using shared memory
3. Analyze the relationship between occupancy and performance
4. Understand warp-level optimizations (shuffle instructions)
5. Calculate the number of active threads at each reduction step

## 1. The Reduction Problem

### Sequential Algorithm

The serial implementation is trivial:

```cpp
int serial_sum(const int* input, int N) {
    int sum = 0;
    for (int i = 0; i < N; i++) {
        sum += input[i];
    }
    return sum;
}
```

**Time Complexity**: O(N)

### The Parallelization Challenge

Reduction has an inherent **data dependency**: each partial sum depends on previous values. This makes it fundamentally different from embarrassingly parallel problems like vector addition.

| Approach | Parallelism | Drawback |
|:---------|:------------|:---------|
| All threads atomic to one location | High contention | Serialized atomics |
| Tree-based reduction | O(log N) steps | Requires synchronization |

## 2. Approach 1: Naive Atomic Reduction

The simplest parallel approach uses atomic operations:

```cpp
__global__ void reduction(const int* input, int* output, int N) {
    int idx = blockDim.x * blockIdx.x + threadIdx.x;

    if (idx >= N) return;

    atomicAdd(output, input[idx]); 
}
```

### Why This Is Inefficient

All threads compete to update a **single memory location**:

```
Thread 0 ---+
Thread 1 ---+
Thread 2 ---+----> atomicAdd(&output, value)  [SERIALIZED!]
Thread 3 ---+
  ...       +
Thread N ---+
```

With millions of threads, this becomes nearly sequential.

## 3. Approach 2: Tree-Based Reduction

The efficient approach uses a **binary tree** pattern:

```
Step 0: [a0, a1, a2, a3, a4, a5, a6, a7]  (8 elements)
         \  /   \  /   \  /   \  /
Step 1:  [s0,   s1,     s2,   s3]         (4 partial sums)
          \    /         \    /
Step 2:   [ss0,          ss1]             (2 partial sums)
             \           /
Step 3:       [final_sum]                 (1 result)
```

**Time Complexity**: O(log N) steps

### Implementation Pattern

```cpp
__global__ void tree_reduction(const int* input, int* output, int N) {
    __shared__ int sdata[256];  // Shared memory for block
    
    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    
    // Load to shared memory
    sdata[tid] = (idx < N) ? input[idx] : 0;
    __syncthreads();
    
    // Tree reduction in shared memory
    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride) {
            sdata[tid] += sdata[tid + stride];
        }
        __syncthreads();
    }
    
    // Write block result
    if (tid == 0) {
        atomicAdd(output, sdata[0]);
    }
}
```

### Exercise 1: Understand Tree Reduction

**Question 1**: For an array of 1024 elements with block size 256, how many reduction steps occur within each block?

Your answer: ______

**Question 2**: After block-level reduction, how many partial sums remain?

Your answer: ______

**Question 3**: Why do we use `__syncthreads()` after each reduction step?

Your answer: ______

## 4. Setup: Generate Test Data

In [ ]:
%%bash
# Generate test cases (low-memory mode for servers)
python3 geninput.py --quick --chunk-size 200000
echo "Test cases generated:"
ls -lh testcases/

## 5. Compile Implementations

In [ ]:
%%bash
# Compile reduction implementation
hipcc -O2 fs_main.hip -o exe_reduction

echo "Compilation complete."

## 6. Run Naive Reduction

In [ ]:
%%bash
echo "=== Small Test (N=10) ==="
./exe_reduction testcases/1.in

echo ""
echo "=== Medium Test (N=1M) ==="
time ./exe_reduction testcases/2.in

## 7. Occupancy Analysis

The test program analyzes how block size affects occupancy and performance.

### Understanding Occupancy

**Occupancy** = Active Warps / Maximum Warps per CU

| Limiting Factor | Description |
|:----------------|:------------|
| **Max Blocks/CU** | Hardware limit on concurrent blocks |
| **Warps** | Total warps exceed CU capacity |
| **Shared Memory** | LDS usage limits concurrent blocks |
| **Registers** | Register usage limits concurrent warps |

### Exercise 2: Analyze Occupancy Results

From the test output, fill in the table:

| Block Size | Warps/Block | Occupancy (%) | Execution Time (ms) |
|:-----------|:------------|:--------------|:--------------------|
| 64 | ______ | ______ | ______ |
| 128 | ______ | ______ | ______ |
| 256 | ______ | ______ | ______ |
| 512 | ______ | ______ | ______ |
| 1024 | ______ | ______ | ______ |

**Question**: Does higher occupancy always mean better performance?

Your answer: ______

## 8. Why Occupancy Is Not Everything

### Key Insight

For reduction, the algorithm efficiency matters more than raw occupancy:

| Factor | Impact on Reduction |
|:-------|:--------------------|
| **Warp Efficiency** | Fewer active threads in later reduction steps |
| **Memory Coalescing** | Initial load pattern affects bandwidth |
| **Instruction Overhead** | More blocks = more final atomic operations |

### Exercise 3: Think About Trade-offs

**Question 1**: With block size 64, how many final atomic operations occur for N = 1M elements?

Calculation: grid_size = ceil(1,000,000 / 64) = ______

**Question 2**: With block size 1024, how many final atomic operations occur?

Calculation: grid_size = ceil(1,000,000 / 1024) = ______

**Question 3**: Why might fewer atomic operations improve performance?

Your answer: ______

## 9. Large Scale Test

In [ ]:
%%bash
# Generate 3.in (100M) if not already present
if [ ! -f testcases/3.in ]; then
    echo "Generating 3.in (this may take a moment)..."
    python3 geninput.py --chunk-size 200000
fi

echo "=== Large Test (N=100M) ==="
time ./exe_reduction testcases/3.in

## 10. Reduction Optimization Strategies

### Level 1: Block-Level Shared Memory Reduction

- Use shared memory for intra-block reduction
- Only one atomic per block to global memory

### Level 2: Warp-Level Reduction

For the last warp, use warp shuffle instructions — no `__syncthreads()` needed:

```cpp
// Tree reduction stops at warpSize, then shuffle finishes it
for (int stride = blockDim.x / 2; stride >= warpSize; stride >>= 1) {
    if (tid < stride) sdata[tid] += sdata[tid + stride];
    __syncthreads();
}
if (tid < warpSize) {
    int val = sdata[tid];
    for (int offset = warpSize / 2; offset > 0; offset >>= 1) {
        val += __shfl_down(val, offset);
    }
}
```

> **Important**: Use the built-in `warpSize` variable, not a hardcoded `32`. RDNA GPUs can run kernels in wave32 or wave64 mode — the compiler chooses per-kernel.

### Level 3: Multi-Element Per Thread

Each thread processes multiple elements before participating in tree reduction:

```cpp
int sum = 0;
for (int i = idx; i < N; i += gridDim.x * blockDim.x) {
    sum += input[i];
}
sdata[tid] = sum;  // Then do tree reduction
```

In [ ]:
%%bash
# Compile the benchmark (all 4 strategies in one binary)
hipcc -O2 benchmark.hip -o exe_benchmark
echo "Benchmark compiled."

In [ ]:
%%bash
echo "=== Benchmark with N=1M (testcases/2.in) ==="
./exe_benchmark testcases/2.in

### TIPS: Wavefront Size on AMD GPUs

| Architecture | Wave Size | Notes |
|:-------------|:----------|:------|
| **GCN / CDNA** (MI100, MI210) | Fixed **64** | All kernels use wave64 |
| **RDNA** (gfx10/gfx11, RX 7000/8000) | **32 or 64** per kernel | Compiler chooses based on heuristics |

On RDNA, the compiler selects wave mode per kernel:
- **wave32** for control-flow-heavy kernels (better single-thread efficiency)
- **wave64** for memory-bound kernels like grid-stride loops (more concurrency to hide latency)

This means **different kernels in the same program can have different wavefront sizes**. Hardcoding `32` or `64` will silently produce wrong results when the compiler picks the other mode.

Always use the built-in `warpSize` variable. It is a compile-time constant — zero overhead:

```cpp
// Correct — works on both wave32 and wave64
for (int stride = blockDim.x / 2; stride >= warpSize; stride >>= 1) {
    if (tid < stride) sdata[tid] += sdata[tid + stride];
    __syncthreads();
}
if (tid < warpSize) {
    int val = sdata[tid];
    for (int offset = warpSize / 2; offset > 0; offset >>= 1)
        val += __shfl_down(val, offset);
}
```

> **Rule: Always use `warpSize`. Never hardcode 32 or 64.**

### Exercise 4: Algorithm Analysis

**Question 1**: In tree reduction, what fraction of threads are idle after the first step?

Your answer: ______

**Question 2**: Why does warp shuffle not require `__syncthreads()`?

Your answer: ______

**Question 3**: How does multi-element per thread improve memory bandwidth utilization?

Your answer: ______

## 11. Summary

### Key Concepts

| Concept | Description |
|:--------|:------------|
| **Reduction** | Combining N elements into a single value |
| **Tree-Based Parallelism** | O(log N) parallel steps |
| **Atomic Contention** | Performance bottleneck with many threads |
| **Occupancy** | Ratio of active to maximum warps |
| **Warp Shuffle** | Intra-warp communication without shared memory |

### Performance Guidelines

| Guideline | Reason |
|:----------|:-------|
| Use tree reduction | Reduces atomic contention |
| Block size 256-512 | Balance between occupancy and efficiency |
| Use warp shuffle for final steps | Avoids shared memory bank conflicts |
| Process multiple elements per thread | Increases arithmetic intensity |

### Key Finding

**High Occupancy != High Performance**

For reduction algorithms, the efficiency of the reduction pattern often matters more than maximizing occupancy.

## 12. Challenge Exercises

### Challenge 1: Implement Tree Reduction

Modify `main.hip` to use shared memory tree reduction instead of naive atomics.

### Challenge 2: Add Warp Shuffle

For the last 64 threads, use warp shuffle intrinsics:
- `__shfl_down()` for AMD GPUs
- Eliminate `__syncthreads()` for intra-warp operations

### Challenge 3: Multi-Pass Reduction

For very large arrays, implement a two-pass reduction:
1. First pass: Reduce to N/block_size partial sums
2. Second pass: Reduce partial sums to final result

### Challenge 4: Benchmark Different Operations

Modify the kernel to compute:
- Maximum value
- Minimum value
- Product (be careful of overflow)

Compare performance differences.

---

## Lab Files Reference

| File | Description |
|:-----|:------------|
| `main.hip` | Student implementation (naive atomic version) |
| `fs_main.hip` | Reference implementation with file I/O |
| `benchmark.hip` | All 4 strategies with GPU timing and comparison |
| `geninput.py` | Test case generator (streaming, low-memory) |
| `Makefile` | Build configuration |
| `README.md` | Problem description and usage instructions |